In [13]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pycontrails import Fleet

In [14]:
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


**TODO clean me up**

In [15]:
# # TODO fuel burn? nox? depends on if I want to have cruise only
# cols = ["fuel_burn", "nox", "NOx", "O3", "CH4", "H2O"]
# cols = ["NOx", "O3", "CH4", "H2O"]

# from cane.utils import mask_by_marker

# mask_by_marker(fleetf, cols)
# mask_by_marker(fleeto, cols)

# dff = fleetf.dataframe
# dfo = fleeto.dataframe

# from cane.utils import mask_by_validity_range

# bounds = [150, 350]
# mask_by_validity_range(fleetf, cols, bounds)
# mask_by_validity_range(fleeto, cols, bounds)

# dff = fleetf.dataframe
# dfo = fleeto.dataframe

In [16]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

In [17]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [18]:
# my_diff = my_diff.query("NOx_optimised != 0").reset_index(drop=True)
# print(my_diff.shape[0], "flights")

In [19]:
meta = pd.read_parquet("../data/filed_meta.parquet")

In [20]:
# WARNING: depending on the filter above this is cruise only / whole trajectory
# Section 3.1
print(my_diff[
          ["fuel_burn_filed", "fuel_burn_optimised", "nox_filed", "nox_optimised"]].sum())  # / meta.flight_id.nunique()

fuel_burn_filed        1.230081e+08
fuel_burn_optimised    1.244774e+08
nox_filed              2.060019e+06
nox_optimised          2.111622e+06
dtype: float64


In [21]:
# normalized does only make sense when I also re-calculate the flown distance
# Section 3.1
(my_diff[["fuel_burn_filed", "fuel_burn_optimised", "nox_filed", "nox_optimised"]].sum() / meta[
    "flown_distance"].sum()).rename("per_flown_distance")

fuel_burn_filed        6.309923
fuel_burn_optimised    6.385296
nox_filed              0.105672
nox_optimised          0.108319
Name: per_flown_distance, dtype: float64

In [22]:
# Section 3.2
print(round(my_diff.query("nox_diff > 0").shape[0] / my_diff.shape[0], 3) * 100, "% flights")
print(
    round(my_diff.query("nox_diff > 0")["nox_diff"].sum() / my_diff.query("nox_diff > 0")["nox_filed"].sum(), 3) * 100,
    "% delta")

95.89999999999999 % flights
2.5 % delta


In [23]:
# Section 3.2
print(round(my_diff.query("NOx_diff > 0").shape[0] / my_diff.shape[0], 3) * 100, "% flights")
print(
    round(my_diff.query("NOx_diff > 0")["NOx_diff"].sum() / my_diff.query("NOx_diff > 0")["NOx_filed"].sum(), 3) * 100,
    "% delta")

69.89999999999999 % flights
2.1 % delta


In [24]:
teoh_nox = 4.49e12 / 60.94e9  # g/km
print(teoh_nox)
print(106 / teoh_nox)
print(108 / teoh_nox)

# EI Lee: 15.14 g/kg(fuel)
# * 6.3 or 6.4 = around 90 g/km

73.67902855267477
1.4386726057906458
1.465817371937639


In [25]:
print(dff.nox_ei.mean())
print(dfo.nox_ei.mean())

0.014554670944845999
0.01469915561087266


In [26]:
distance = meta["flown_distance"].sum()  # km
print(dff.nvpm_ei_m.mean() * 1e3, dfo.nvpm_ei_m.mean() * 1e3, "nvPM EI m")  # from kg / kg(fuel) to g/kg
print(dff.nvpm_ei_n.mean() / 1e15, dfo.nvpm_ei_n.mean() / 1e15, 1.115 / 1.104, "nvPM EI n")  # / kg(fuel)
print(dff.nvpm_mass.sum(), dfo.nvpm_mass.sum(), 7892 / 7728)  # kg
print(dff.nvpm_number.sum(), dfo.nvpm_number.sum(), 9.81 / 9.55)  # -
print(7728 * 1e3 / distance, 7892 * 1e3 / distance, 0.4048153619449876 / 0.39642095491550416)
print(9.545647252292154e+22 / distance / 1e15, 9.809877818314508e+22 / distance / 1e15, 5.032 / 4.899,
      "number per distance")

# Teoh 2024
print(21.4e9 / 60.94e9, 0.396 / 0.351, 0.405 / 0.351)  # mass per distance
print(2.83e26 / 60.94e9 / 1e15, 4.899 / 4.644, 5.032 / 4.644)  # number per distance

0.07323674658963993 0.0734330048778605 nvPM EI m
1.1043396879548946 1.1154158927301243 1.009963768115942 nvPM EI n
7727.984281164688 7891.628116762056 1.0212215320910973
9.545647252292152e+22 9.80987781831451e+22 1.0272251308900524
0.39642176124164036 0.40483443836943916 1.0211754876360475
4.896612701792288 5.032154557843604 1.0271483976321698 number per distance
0.35116508040695765 1.1282051282051284 1.153846153846154
4.643912044634066 1.0549095607235142 1.0835486649440138


In [27]:
print(dff.ef.sum(), dfo.ef.sum(), 0.970 / 1.717)
print(dff.ef.sum() / 4112 / 1e14, dfo.ef.sum() / 4112 / 1e14, 2.358 / 4.175)
print(dff.ef.sum() / distance / 1e10)
print(dfo.ef.sum() / distance / 1e10)

# 4.1 * x = 2.3
# 1 - 2.3 / 4.1

1.716572655120809e+18 9.695704642231345e+17 0.564938846825859
4.174544394749049 2.357904825445366 0.5647904191616767
8.805470435329063
4.973587358633667


In [28]:
# Martin Frias 2024
print(0.874e18 / 84839 / 1e14, "cost optimal")  # J per flight
print(0.236e18 / 84839 / 1e14, "climate optimal")
print(0.3e18 / 2260 / 1e14, "big hits", 1.327 / 2.358, 1.327 / 4.175)  # this is wrong; see Dennis explanation

0.10301865887150957 cost optimal
0.02781739530168908 climate optimal
1.3274336283185841 big hits 0.5627650551314674 0.3178443113772455


In [29]:
# n flights
4112 / 40221182

0.00010223468817997442

In [30]:
# GAIA vs others
print(283 / 60.94)
print(320 / 60.94)
print(297 / 60.94, 6.3 / 4.874, 6.4 / 4.874)

4.6439120446340665
5.2510666229077785
4.873646209386282 1.2925728354534265 1.313089864587608


In [31]:
from cane.utils import mask_by_marker

mask_by_marker(fleetf, ["NOx", "O3", "CH4", "H2O"])  # ["nox", "NOx", "O3", "CH4", "H2O"])
mask_by_marker(fleeto, ["NOx", "O3", "CH4", "H2O"])  #["nox", "NOx", "O3", "CH4", "H2O"])

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O


In [32]:
def haul_type_by_distance_ectl(df):
    # ("long-haul": $>\SI{4000}{\kilo\meter}$, "medium-haul": 1500--\SI{4000}{\kilo\meter}, "short-haul": $<\SI{1500}{\kilo\meter}$
    df["haul_type"] = np.where(df.flown_distance > 4000, "Long", "Medium")
    df["haul_type"] = np.where(df.flown_distance < 1500, "Short", df["haul_type"])
    return df

In [33]:
# from src.flight.trajectory import flight_haul_type_by_time, flown_distance
# meta2 = flight_haul_type_by_time(meta, classification="IATA")
meta2 = haul_type_by_distance_ectl(meta)

dff2 = pd.merge(dff, meta2[["flight_id", "haul_type"]], on="flight_id")
dfo2 = pd.merge(dfo, meta2[["flight_id", "haul_type"]], on="flight_id")

In [34]:
print(dff.NOx.sum() / distance)
print(dfo.NOx.sum() / distance)

print(dff.H2O.sum() / distance * 1e3)
print(dfo.H2O.sum() / distance * 1e3)
print(0.6078837285873402 / 0.6149718901345674)

10.145363681113205
10.224648819953032
614.9718901345676
607.8837285873403
0.988474007249866


In [35]:
meta2.groupby("haul_type")["flown_distance"].sum().reset_index()

,haul_type,flown_distance
0,Long,1.574406e+07
1,Medium,2.898554e+06
2,Short,8.517736e+05


In [36]:
dff2.groupby("haul_type")[["NOx", "H2O"]].sum().reset_index()

,haul_type,NOx,H2O
0,Long,1.780483e+08,1.109790e+07
1,Medium,1.367884e+07,7.103406e+05
2,Short,6.050500e+06,1.802560e+05


In [37]:
filed = pd.merge(
    meta2.groupby("haul_type")["flown_distance"].sum().reset_index(),
    dff2.groupby("haul_type")[["NOx", "H2O"]].sum().reset_index(),
    on="haul_type",
)
filed["NOx_per_distance"] = filed.NOx / filed.flown_distance  # kg / km
filed["H2O_per_distance"] = filed.H2O / filed.flown_distance  # kg / km
filed

,haul_type,flown_distance,NOx,H2O,NOx_per_distance,H2O_per_distance
0,Long,1.574406e+07,1.780483e+08,1.109790e+07,11.308920,0.704895
1,Medium,2.898554e+06,1.367884e+07,7.103406e+05,4.719193,0.245067
2,Short,8.517736e+05,6.050500e+06,1.802560e+05,7.103413,0.211624


In [38]:
optimised = pd.merge(
    meta2.groupby("haul_type")["flown_distance"].sum().reset_index(),
    dfo2.groupby("haul_type")[["NOx", "H2O"]].sum().reset_index(),
    on="haul_type",
)
optimised["NOx_per_distance"] = optimised.NOx / optimised.flown_distance  # kg / km
optimised["H2O_per_distance"] = optimised.H2O / optimised.flown_distance  # kg / km
optimised

,haul_type,flown_distance,NOx,H2O,NOx_per_distance,H2O_per_distance
0,Long,1.574406e+07,1.795346e+08,1.097946e+07,11.403320,0.697372
1,Medium,2.898554e+06,1.373005e+07,6.956377e+05,4.736859,0.239995
2,Short,8.517736e+05,6.058671e+06,1.752201e+05,7.113006,0.205712


In [39]:
print(4.736859 / 4.719193)
print(7.113006 / 7.103413)

1.0037434366426634
1.0013504775802844


In [40]:
# Dennis (NOx)
print((10.3 * 19 + 9 * 10 + 0.1 * 0.4) / (10.3 + 9 + 0.1), "long")
print((1.8 * 7 + 28.9 * 4 + 0.2 * 0.2) / (1.8 + 28.9 + 0.2), "medium")
print((0.4 * 4 + 37.1 * 3 + 12.3 * 0.2) / (0.4 + 37.1 + 12.3), "short")
print((10.3 * 19 + 9 * 10 + 0.1 * 0.4 + 1.8 * 7 + 28.9 * 4 + 0.2 * 0.2 + 0.4 * 4 + 37.1 * 3 + 12.3 * 0.2) / (
        10.3 + 9 + 0.1 + 1.8 + 28.9 + 0.2 + 0.4 + 37.1 + 12.3), "all")

print("evt. * 1.045 to account for efficacy")

14.728865979381444 long
4.150161812297735 medium
2.3164658634538156 short
5.288111888111889 all
evt. * 1.045 to account for efficacy


In [41]:
print(dff.fuel_burn.sum() / 1e6)  # mega
print(dfo.fuel_burn.sum() / 1e6)

print(dff.co2.sum() / 1e6)
print(dfo.co2.sum() / 1e6)

print(dff.nox.sum() / 1e6)
print(dfo.nox.sum() / 1e6)

print(dff.co2.sum() / distance)
print(dfo.co2.sum() / distance)

123.00809651666546
124.47743885486715
388.5825768962306
393.22422934390744
2.0600188332211005
2.111621807763269
19.933047298268487
20.1711492701085


In [42]:
# Teoh Global 2024
print(0.164e8 * 1e3 / 1e10)  # from J / m to J / km

print(dff.ef.sum() / distance / 1e10)
print(dfo.ef.sum() / distance / 1e10)

1.64
8.805470435329063
4.973587358633667


In [43]:
# Teoh global CO2
885e9 / 60.94e9

14.522481128979324

In [44]:
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [45]:
# TODO fuel burn? nox? depends on if I want to have cruise only
cols = ["fuel_burn", "nox", "NOx", "O3", "CH4", "H2O"]
cols = ["NOx", "O3", "CH4", "H2O"]

from cane.utils import mask_by_marker

mask_by_marker(fleetf, cols)
mask_by_marker(fleeto, cols)

dff = fleetf.dataframe
dfo = fleeto.dataframe

from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, cols, bounds)
mask_by_validity_range(fleeto, cols, bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [46]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

dff["NOx_H2O"] = dff["NOx"] + dff["H2O"]
dfo["NOx_H2O"] = dfo["NOx"] + dfo["H2O"]

In [47]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "NOx", "CoCiP", "CO2", "H2O", "Total", "CO2_CoCiP",
         "NOx_H2O"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [48]:
# Table v2 (based on Volker's input)
metaf = pd.read_parquet("../data/filed_meta.parquet")
metao = pd.read_parquet("../data/optimised_meta.parquet")

# metadiff = df_diff(metaf[["flight_id", "flown_distance", "flight_time"]],
#                    metao[["flight_id", "flown_distance", "flight_time"]],
#                    keep_originals=True)

In [49]:
print(metaf.flight_time.sum(), metao.flight_time.sum(), "")
print(round(metaf.flown_distance.sum() / 1e6, 1), round(metao.flown_distance.sum() / 1e6, 1), "x 10^6 km")
print(round(my_diff.fuel_burn_filed.sum() / 1e6, 3), round(my_diff.fuel_burn_optimised.sum() / 1e6, 3), "Gg")
print(round(my_diff["nox_filed"].sum() / 1e6, 3), round(my_diff["nox_optimised"].sum() / 1e6, 3), "Gg")
print((2.112 - 2.06) / 2.06 * 100)

950 days 15:08:44.020000 948 days 16:26:32.006000 
19.5 19.5 x 10^6 km
123.008 124.477 Gg
2.06 2.112 Gg
2.5242718446601966


In [52]:
# WARNING: depending on the filter above this is cruise only / whole trajectory
print(my_diff[
          ["fuel_burn_filed", "fuel_burn_optimised", "nox_filed", "nox_optimised"]].sum())  # / meta.flight_id.nunique()

fuel_burn_filed        1.230081e+08
fuel_burn_optimised    1.244774e+08
nox_filed              2.060019e+06
nox_optimised          2.111622e+06
dtype: float64


In [51]:
# normalized does only make sense when I also re-calculate the flown distance
(my_diff[["fuel_burn_filed", "fuel_burn_optimised", "nox_filed", "nox_optimised"]].sum() / metaf[
    "flown_distance"].sum()).rename("per_flown_distance")

fuel_burn_filed        6.309923
fuel_burn_optimised    6.385296
nox_filed              0.105672
nox_optimised          0.108319
Name: per_flown_distance, dtype: float64